# 03 — Counterfactual Analysis

Visualize what-if trajectory predictions under causal interventions.

In [ ]:
import sys; sys.path.insert(0, '..')
import torch
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
from omegaconf import OmegaConf
from transformers import AutoTokenizer
from data.loaders.physicalai_av_dataset import build_dataloader
from models.se_cvla import SECVLA

cfg = OmegaConf.load('../configs/base.yaml')
tokenizer = AutoTokenizer.from_pretrained(cfg.model.encoder.language_backbone, trust_remote_code=True)
model = SECVLA.from_pretrained('../outputs/stage3/best.ckpt', cfg)
model.eval()
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = model.to(device)

In [ ]:
# Load one batch
loader = build_dataloader(cfg.data, split='val', tokenizer=tokenizer)
batch = next(iter(loader))
batch = batch.to(device)

# Factual prediction
with torch.no_grad():
    factual_out = model.predict(batch, num_traj_samples=8)

factual_traj = factual_out['best_trajectory'][0].cpu().numpy()  # (H, 2)
print(f'Factual trajectory shape: {factual_traj.shape}')

In [ ]:
# Counterfactual: what-if we remove the leading vehicle (variable 0)?
with torch.no_grad():
    cf_trajs = model.what_if(
        batch=batch,
        variable_idx=0,
        value=torch.zeros(batch.images.shape[0], 64, device=device),
        num_samples=8,
    )  # (B, 8, H, 2)

cf_mean = cf_trajs[0].mean(dim=0).cpu().numpy()  # (H, 2)
gt_traj = batch.trajectory_gt[0].cpu().numpy()

# Plot
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
for ax, title, traj, color in [
    (axes[0], 'Factual Prediction', factual_traj, 'blue'),
    (axes[1], 'Counterfactual: Remove Agent 0', cf_mean, 'orange'),
]:
    ax.plot(gt_traj[:, 0], gt_traj[:, 1], 'g--', label='GT', linewidth=2)
    ax.plot(traj[:, 0], traj[:, 1], color=color, linewidth=2, label=title, marker='o', markersize=3)
    ax.plot(0, 0, 'r*', markersize=12, label='Ego')
    ax.legend(); ax.set_title(title); ax.axis('equal'); ax.grid(True)
    ax.set_xlabel('x (m)'); ax.set_ylabel('y (m)')

plt.suptitle('Factual vs. Counterfactual Trajectories', fontsize=13)
plt.tight_layout(); plt.show()

In [ ]:
# Show all CF samples + factual spread
fig, ax = plt.subplots(figsize=(8, 10))

# Factual samples
for k in range(factual_out['trajectories'].shape[1]):
    t = factual_out['trajectories'][0, k].cpu().numpy()
    ax.plot(t[:, 0], t[:, 1], 'b-', alpha=0.3, linewidth=1)

# CF samples
for k in range(cf_trajs.shape[1]):
    t = cf_trajs[0, k].cpu().numpy()
    ax.plot(t[:, 0], t[:, 1], 'r-', alpha=0.3, linewidth=1)

# GT
ax.plot(gt_traj[:, 0], gt_traj[:, 1], 'g--', linewidth=2, label='GT')
ax.plot(0, 0, 'k*', markersize=14, label='Ego t=0')

blue_patch = mpatches.Patch(color='blue', alpha=0.4, label='Factual samples')
red_patch  = mpatches.Patch(color='red',  alpha=0.4, label='CF samples (agent removed)')
ax.legend(handles=[blue_patch, red_patch, 
    mpatches.Patch(color='green', label='GT'),
    mpatches.Patch(color='black', label='Ego')])
ax.set_title('Trajectory Sample Distributions: Factual vs. Counterfactual')
ax.set_xlabel('x (m)'); ax.set_ylabel('y (m)')
ax.axis('equal'); ax.grid(True); plt.tight_layout(); plt.show()